In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import plotly.graph_objects as go
import plotly_express as px

In [30]:
INSTRUMENT_MAP = {
    # S&P 500 — all variants map to same instrument
    "ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE" :    "SP500",
    "E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE" :                    "SP500",
    "E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",
    "MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",
    "S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE" :     "SP500",
    "S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE" :              "SP500",
    "S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE" :    "SP500",
    "S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE" :               "SP500",
    "S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",

    'BITCOIN - CHICAGO MERCANTILE EXCHANGE':                            'BTC',
    'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC':              'BTC',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE':                      'BTC',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC':              'BTC',

    "ULTRA 10-YEAR U.S. T-NOTES - CHICAGO BOARD OF TRADE" :             "US10Y",
    "ULTRA UST 10Y - CHICAGO BOARD OF TRADE" :                          "US10Y",
    "UST 10Y NOTE - CHICAGO BOARD OF TRADE" :                           "US10Y",
    "10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE" :            "US10Y",
    
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE' :                      "3 Month US",
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE' :                           "3 Month US",


    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE' :                           '1 Month US',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE' :                      '1 Month US',

    "E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE":          "russel",
    "EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE":          "russel",
    "EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE":     "russel",
    "MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE":     "russel",
    "RUSSELL 1000 VALUE INDEX MINI - ICE FUTURES U.S.":                 "russel",
    "RUSSELL 2000 ANNUAL DIVIDEND  - CHICAGO MERCANTILE EXCHANGE":      "russel",
    "RUSSELL 2000 MINI INDEX FUTURE - ICE FUTURES U.S.":                "russel",
    "RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE":                     "russel",

    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE':                      'GBP',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE':             'GBP',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE':        'GBP',

    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE':                       'YEN',

    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':                    "CAD",
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE' :                       "CHF",

    "MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE" :     "NASDAQ",
    "NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE" :                       "NASDAQ",
    "NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE" :           "NASDAQ",
    "NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE" :     "NASDAQ",


    "2-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE": "2Y",
    "UST 2Y NOTE - CHICAGO BOARD OF TRADE":                "2Y",
    
    "5-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE": "5Y",
    "UST 5Y NOTE - CHICAGO BOARD OF TRADE":                "5Y",
    
    "10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE": "7Y",
    "UST 10Y NOTE - CHICAGO BOARD OF TRADE":                "7Y",
    
    "ULTRA 10-YEAR U.S. T-NOTES - CHICAGO BOARD OF TRADE": "10Y",
    "ULTRA UST 10Y - CHICAGO BOARD OF TRADE":              "10Y",
    
    "U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE": "20Y",
    "UST BOND - CHICAGO BOARD OF TRADE":            "20Y",
    
    "ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE": "30Y",
    "ULTRA UST BOND - CHICAGO BOARD OF TRADE":            "30Y",
    "ULTRA US T BOND - CHICAGO BOARD OF TRADE":           "30Y",
    
    
    
    }

In [ ]:
TICKER_MAP = {
    'SP500':        '^GSPC',
    'BTC':          'BTC-USD',
    'US10Y':        'IEF',
    '3 Month US' :  'SHV',
    '1 Month US' :  'SHV',
    'russel':       'IWM',
    'GBP':          'GBPUSD=X',
    'YEN':          'JPY=X',
    'CAD' :         'CAD=X',
    'CHF' :         'CHF=X',
    'NASDAQ':       'QQQ',



    
}

In [31]:
def download_cot_data(years, report_type='combined'):
    """
    Downloads raw CFTC Traders in Financial Futures (TFF) data.

    Args:
        years       : list of int — e.g. [2020, 2021, 2022]
        report_type : 'futures' or 'combined' (futures + options)

    Returns:
        pd.DataFrame — raw unmodified TFF data
    """
    prefix_map = {
        'futures':  'fut_fin_xls',
        'combined': 'com_fin_xls',
    }
    assert report_type in prefix_map, f"report_type must be one of {list(prefix_map.keys())}"
    prefix = prefix_map[report_type]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer':    'https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm',
    }

    dfs = []
    for year in years:
        url = f"https://www.cftc.gov/files/dea/history/{prefix}_{year}.zip"
        print(f"Fetching {report_type} COT {year} from {url}...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            zf  = zipfile.ZipFile(io.BytesIO(response.content))
            xls = zf.open(zf.namelist()[0]).read()
            dfs.append(pd.read_excel(io.BytesIO(xls), engine='xlrd'))
            print(f"  ✓ {year} fetched successfully")
        except requests.HTTPError as e:
            print(f"  ✗ HTTP error for {year}: {e}")
        except zipfile.BadZipFile:
            print(f"  ✗ Bad zip file for {year}")
        except Exception as e:
            print(f"  ✗ Unexpected error for {year}: {e}")

    if not dfs:
        raise RuntimeError("No data fetched — check years and network connection")

    return pd.concat(dfs, ignore_index=True)

In [6]:
def wavg(group, conc_cols):
        weights = group['Open_Interest_All']
        total = weights.sum()
        if total == 0:
            return pd.Series({c: 0.0 for c in conc_cols})
        return pd.Series({c: (group[c] * weights).sum() / total for c in conc_cols})

In [7]:
def clean_financial_cot_data(df, instrument_map):
    ''' Dealer — financial institutions/dealers (often on the other side of client trades)
        Asset_Mgr — institutional investors like pension funds, mutual funds
        Lev_Money — leveraged money (hedge funds, CTAs)
        Other_Rept — other reportable traders
        NonRept — non-reportable (small traders below reporting thresholds)
        Traders - it's counting number of traders, knowing 47 dealers are long tells you less than knowing their total position size
        Concentration Report - Shows what % of open interest the top 4 and top 8 traders control'''
    
    # take only the columns we need
    df = df[[
        'Market_and_Exchange_Names', 
        'Report_Date_as_MM_DD_YYYY', 
        'Open_Interest_All', 

        # Dealer
        'Dealer_Positions_Long_All', 'Dealer_Positions_Short_All', 'Dealer_Positions_Spread_All',
        # Asset Manager
        'Asset_Mgr_Positions_Long_All', 'Asset_Mgr_Positions_Short_All', 'Asset_Mgr_Positions_Spread_All',
        # Hedge Fund / Leveraged
        'Lev_Money_Positions_Long_All', 'Lev_Money_Positions_Short_All', 'Lev_Money_Positions_Spread_All',
        # Other Reportable
        'Other_Rept_Positions_Long_All', 'Other_Rept_Positions_Short_All', 'Other_Rept_Positions_Spread_All',
        # Non-Reportable
        'NonRept_Positions_Long_All', 'NonRept_Positions_Short_All',

        # Concentration — top 4 and top 8 traders
        'Conc_Gross_LE_4_TDR_Long_All', 'Conc_Gross_LE_4_TDR_Short_All',
        'Conc_Gross_LE_8_TDR_Long_All', 'Conc_Gross_LE_8_TDR_Short_All',
    ]]


    df['Report_Date_as_MM_DD_YYYY'] = pd.to_datetime(df['Report_Date_as_MM_DD_YYYY'])
    df = df.sort_values('Report_Date_as_MM_DD_YYYY')

    df['Instrument'] = df['Market_and_Exchange_Names'].map(instrument_map)
    df = df[df['Instrument'].notna()].copy()

    conc_cols = [
        'Conc_Gross_LE_4_TDR_Long_All', 'Conc_Gross_LE_4_TDR_Short_All',
        'Conc_Gross_LE_8_TDR_Long_All', 'Conc_Gross_LE_8_TDR_Short_All',
    ]

    cols_to_sum = df.columns.values.tolist()
    cols_to_sum.remove('Market_and_Exchange_Names')
    cols_to_sum.remove('Report_Date_as_MM_DD_YYYY')
    cols_to_sum.remove('Instrument')
    for c in conc_cols:
        cols_to_sum.remove(c)

    # Sum position columns
    df_summed = (
        df.groupby(['Instrument', 'Report_Date_as_MM_DD_YYYY'])[cols_to_sum]
        .sum()
        .reset_index()
    )

    # Weighted average concentration by OI
    

    df_conc = (
        df.groupby(['Instrument', 'Report_Date_as_MM_DD_YYYY'])
        .apply(wavg, conc_cols)
        .reset_index()
    )

    df = df_summed.merge(df_conc, on=['Instrument', 'Report_Date_as_MM_DD_YYYY'])


    return df

In [8]:
def feature_engineering(cot_clean):
    cot_clean["Dealer Net"] = cot_clean['Dealer_Positions_Long_All'] - cot_clean['Dealer_Positions_Short_All']
    cot_clean["Asset Manager Net"] = cot_clean['Asset_Mgr_Positions_Long_All'] - cot_clean['Asset_Mgr_Positions_Short_All'] 
    cot_clean["Levered Net"] = cot_clean['Lev_Money_Positions_Long_All'] - cot_clean['Lev_Money_Positions_Short_All']
    cot_clean["Other Rept Positions Net"] = cot_clean['Other_Rept_Positions_Long_All'] - cot_clean['Other_Rept_Positions_Short_All']
    cot_clean["Non Rept Positions Net"] = cot_clean['NonRept_Positions_Long_All'] - cot_clean['NonRept_Positions_Short_All']
    
    # In feature_engineering:
    cot_clean["Dealer Net Pct OI"] = cot_clean["Dealer Net"] / cot_clean["Open_Interest_All"]
    cot_clean["AM Net Pct OI"] = cot_clean["Asset Manager Net"] / cot_clean["Open_Interest_All"]
    cot_clean["HF Net Pct OI"] = cot_clean["Levered Net"] / cot_clean["Open_Interest_All"]
    cot_clean["Other Rept Positions Net Pct OI"] = cot_clean['Other Rept Positions Net']/cot_clean["Open_Interest_All"]
    cot_clean["Non Rept Positions Net Pct OI"] = cot_clean['Non Rept Positions Net']/cot_clean["Open_Interest_All"]


    return cot_clean

In [9]:
def get_align_prices(cot_clean):
    instrument_list = cot_clean['Instrument'].unique().tolist()
    tickers_list = [TICKER_MAP[inst] for inst in instrument_list if inst in TICKER_MAP]
    tickers_list
    prices = yf.download(tickers_list,start = '2018-01-01' )['Close']
    prices = prices.ffill()
    prices = prices.bfill()
    prices
    all_dfs = []
    for instr in instrument_list:
        cot_specific_instrument = cot_clean[cot_clean['Instrument'] ==instr ]
        cot_specific_instrument = cot_specific_instrument.set_index('Report_Date_as_MM_DD_YYYY')
        ticker_name = TICKER_MAP[instr]
        price_specific_instrument = pd.DataFrame(prices[ticker_name])
        merged_df = cot_specific_instrument.merge(price_specific_instrument, left_index=True, right_index=True, how = 'left')
        merged_df = merged_df.rename(columns={ticker_name: 'Price'})
        all_dfs.append(merged_df)
    cot_data = pd.concat(all_dfs)
    cot_data = cot_data.reset_index()
    return cot_data
    

In [10]:
cot_raw = download_cot_data([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])
cot_clean = clean_financial_cot_data(cot_raw, INSTRUMENT_MAP)
cot_clean
cot_clean = feature_engineering(cot_clean)
cot_clean = get_align_prices(cot_clean)
cot_clean

Fetching combined COT 2018 from https://www.cftc.gov/files/dea/history/com_fin_xls_2018.zip...
  ✓ 2018 fetched successfully
Fetching combined COT 2019 from https://www.cftc.gov/files/dea/history/com_fin_xls_2019.zip...
  ✓ 2019 fetched successfully
Fetching combined COT 2020 from https://www.cftc.gov/files/dea/history/com_fin_xls_2020.zip...
  ✓ 2020 fetched successfully
Fetching combined COT 2021 from https://www.cftc.gov/files/dea/history/com_fin_xls_2021.zip...
  ✓ 2021 fetched successfully
Fetching combined COT 2022 from https://www.cftc.gov/files/dea/history/com_fin_xls_2022.zip...
  ✓ 2022 fetched successfully
Fetching combined COT 2023 from https://www.cftc.gov/files/dea/history/com_fin_xls_2023.zip...
  ✓ 2023 fetched successfully
Fetching combined COT 2024 from https://www.cftc.gov/files/dea/history/com_fin_xls_2024.zip...
  ✓ 2024 fetched successfully
Fetching combined COT 2025 from https://www.cftc.gov/files/dea/history/com_fin_xls_2025.zip...
  ✓ 2025 fetched successfully


/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/96025408.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Report_Date_as_MM_DD_YYYY'] = pd.to_datetime(df['Report_Date_as_MM_DD_YYYY'])
/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/96025408.py:63: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(wavg, conc_cols)
/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/1065241557.py:5: FutureWarning: YF.d

,Report_Date_as_MM_DD_YYYY,Instrument,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,...,Asset Manager Net,Levered Net,Other Rept Positions Net,Non Rept Positions Net,Dealer Net Pct OI,AM Net Pct OI,HF Net Pct OI,Other Rept Positions Net Pct OI,Non Rept Positions Net Pct OI,Price
0,2018-07-03,1 Month US,5921,3312,609,373,0,0,0,1,...,0,-2793,0,90,0.456511,0.000000,-0.471711,0.000000,0.015200,90.289513
1,2018-07-10,1 Month US,6670,3481,683,800,0,0,0,35,...,0,-2658,0,-140,0.419490,0.000000,-0.398501,0.000000,-0.020990,90.305885
2,2018-07-17,1 Month US,8879,5557,2356,738,0,0,0,28,...,0,-3205,-25,29,0.360514,0.000000,-0.360964,-0.002816,0.003266,90.338654
3,2018-07-24,1 Month US,12655,7410,3562,1061,0,0,0,748,...,0,-4025,102,75,0.304070,0.000000,-0.318056,0.008060,0.005927,90.363174
4,2018-07-31,1 Month US,14365,9576,4733,794,0,0,0,242,...,0,-4944,0,101,0.337139,0.000000,-0.344170,0.000000,0.007031,90.395950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4688,2026-03-17,russel,586922,144519,109868,40897,178574,134742,28596,81044,...,43832,-58919,-4348,-15217,0.059039,0.074681,-0.100386,-0.007408,-0.025927,250.050003
4689,2026-03-24,russel,478081,122590,95247,8324,166950,135445,27855,88922,...,31505,-53774,746,-5819,0.057193,0.065899,-0.112479,0.001560,-0.012172,248.779999
4690,2026-03-31,russel,472964,140220,103142,7998,160678,148025,28789,74459,...,12653,-41710,-4092,-3929,0.078395,0.026753,-0.088189,-0.008652,-0.008307,248.000000
4691,2026-04-07,russel,470795,141536,99737,7783,153602,149223,28589,79994,...,4379,-34970,-4271,-6936,0.088784,0.009301,-0.074279,-0.009072,-0.014733,252.910004


In [11]:
def find_dominant_contracts(df, instrument_name):
    """
    For a given instrument, shows which contract names exist,
    how many weeks each was dominant, and the date range of dominance.
    
    Use this BEFORE updating INSTRUMENT_MAP to understand what to include.
    
    Args:
        df              : raw COT DataFrame
        instrument_name : str — search term e.g. 'EURO FX', 'NASDAQ', 'BITCOIN'
        top_n           : int — how many contracts to show
    
    Returns:
        pd.DataFrame — contract names, dominance count, date ranges
    """
    # Find all contracts matching the search term and print the variant names
    df_filtered = df[df['Market_and_Exchange_Names'].str.contains(instrument_name)]
    df_filtered = df_filtered[['Market_and_Exchange_Names', 'Report_Date_as_MM_DD_YYYY', 'Open_Interest_All']]
    df_filtered_pivot = df_filtered.pivot_table(
    index='Report_Date_as_MM_DD_YYYY',
    columns='Market_and_Exchange_Names',
    values='Open_Interest_All',
    aggfunc='first' ) # in case of duplicates, take first

    
    variant_names = (df_filtered_pivot.columns.values.tolist())
    print(f"Variants: For {instrument_name}: ") 
    for i in variant_names:
        print(i)
    if len(variant_names) == 0:
        print('Probably Searched Wrong')

    df_filtered_pivot['Total'] = df_filtered_pivot.sum(axis=1, )

    df_filtered_pivot_proportions = df_filtered_pivot.div(df_filtered_pivot['Total'], axis=0) * 100
    df_filtered_pivot_proportions = df_filtered_pivot_proportions.drop(columns='Total')
    return df_filtered_pivot_proportions

In [12]:
cot_raw

,Market_and_Exchange_Names,As_of_Date_In_Form_YYMMDD,Report_Date_as_MM_DD_YYYY,CFTC_Contract_Market_Code,CFTC_Market_Code,CFTC_Region_Code,CFTC_Commodity_Code,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,...,Conc_Gross_LE_4_TDR_Short_All,Conc_Gross_LE_8_TDR_Long_All,Conc_Gross_LE_8_TDR_Short_All,Conc_Net_LE_4_TDR_Long_All,Conc_Net_LE_4_TDR_Short_All,Conc_Net_LE_8_TDR_Long_All,Conc_Net_LE_8_TDR_Short_All,Contract_Units,CFTC_SubGroup_Code,FutOnly_or_Combined
0,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181231,2018-12-31,090741,CME,0,90,190149,58230,393,...,19.3,49.6,30.8,29.9,19.3,46.9,29.7,"(CONTRACTS OF CAD 100,000)",F10,Combined
1,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181224,2018-12-24,090741,CME,0,90,179754,52234,381,...,16.6,47.4,28.1,30.3,16.4,44.5,27.0,"(CONTRACTS OF CAD 100,000)",F10,Combined
2,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181218,2018-12-18,090741,CME,0,90,209891,52401,3956,...,19.1,51.1,28.4,25.8,11.9,42.9,20.2,"(CONTRACTS OF CAD 100,000)",F10,Combined
3,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181211,2018-12-11,090741,CME,0,90,176889,35597,5632,...,16.8,46.9,27.9,25.7,14.5,38.1,24.1,"(CONTRACTS OF CAD 100,000)",F10,Combined
4,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181204,2018-12-04,090741,CME,0,90,172849,34034,5397,...,15.8,47.1,28.1,27.1,14.1,37.1,24.5,"(CONTRACTS OF CAD 100,000)",F10,Combined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23666,BBG COMMODITY - CHICAGO BOARD OF TRADE,260127,2026-01-27,221602,CBT,0,221,188271,125275,172650,...,89.4,89.8,96.8,82.8,89.4,87.9,95.4,($100 X INDEX),F90,Combined
23667,BBG COMMODITY - CHICAGO BOARD OF TRADE,260120,2026-01-20,221602,CBT,0,221,189350,125416,171958,...,88.5,89.7,96.0,82.6,88.5,87.7,94.8,($100 X INDEX),F90,Combined
23668,BBG COMMODITY - CHICAGO BOARD OF TRADE,260113,2026-01-13,221602,CBT,0,221,189092,125364,171894,...,88.6,89.7,96.2,82.7,88.6,87.8,95.0,($100 X INDEX),F90,Combined
23669,BBG COMMODITY - CHICAGO BOARD OF TRADE,260106,2026-01-06,221602,CBT,0,221,204353,124995,187085,...,90.5,88.1,96.5,77.6,90.5,86.5,95.0,($100 X INDEX),F90,Combined


In [32]:
cot_raw = download_cot_data([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])
cot_raw

Fetching combined COT 2018 from https://www.cftc.gov/files/dea/history/com_fin_xls_2018.zip...
  ✓ 2018 fetched successfully
Fetching combined COT 2019 from https://www.cftc.gov/files/dea/history/com_fin_xls_2019.zip...
  ✓ 2019 fetched successfully
Fetching combined COT 2020 from https://www.cftc.gov/files/dea/history/com_fin_xls_2020.zip...
  ✓ 2020 fetched successfully
Fetching combined COT 2021 from https://www.cftc.gov/files/dea/history/com_fin_xls_2021.zip...
  ✓ 2021 fetched successfully
Fetching combined COT 2022 from https://www.cftc.gov/files/dea/history/com_fin_xls_2022.zip...
  ✓ 2022 fetched successfully
Fetching combined COT 2023 from https://www.cftc.gov/files/dea/history/com_fin_xls_2023.zip...
  ✓ 2023 fetched successfully
Fetching combined COT 2024 from https://www.cftc.gov/files/dea/history/com_fin_xls_2024.zip...
  ✓ 2024 fetched successfully
Fetching combined COT 2025 from https://www.cftc.gov/files/dea/history/com_fin_xls_2025.zip...
  ✓ 2025 fetched successfully


,Market_and_Exchange_Names,As_of_Date_In_Form_YYMMDD,Report_Date_as_MM_DD_YYYY,CFTC_Contract_Market_Code,CFTC_Market_Code,CFTC_Region_Code,CFTC_Commodity_Code,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,...,Conc_Gross_LE_4_TDR_Short_All,Conc_Gross_LE_8_TDR_Long_All,Conc_Gross_LE_8_TDR_Short_All,Conc_Net_LE_4_TDR_Long_All,Conc_Net_LE_4_TDR_Short_All,Conc_Net_LE_8_TDR_Long_All,Conc_Net_LE_8_TDR_Short_All,Contract_Units,CFTC_SubGroup_Code,FutOnly_or_Combined
0,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181231,2018-12-31,090741,CME,0,90,190149,58230,393,...,19.3,49.6,30.8,29.9,19.3,46.9,29.7,"(CONTRACTS OF CAD 100,000)",F10,Combined
1,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181224,2018-12-24,090741,CME,0,90,179754,52234,381,...,16.6,47.4,28.1,30.3,16.4,44.5,27.0,"(CONTRACTS OF CAD 100,000)",F10,Combined
2,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181218,2018-12-18,090741,CME,0,90,209891,52401,3956,...,19.1,51.1,28.4,25.8,11.9,42.9,20.2,"(CONTRACTS OF CAD 100,000)",F10,Combined
3,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181211,2018-12-11,090741,CME,0,90,176889,35597,5632,...,16.8,46.9,27.9,25.7,14.5,38.1,24.1,"(CONTRACTS OF CAD 100,000)",F10,Combined
4,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,181204,2018-12-04,090741,CME,0,90,172849,34034,5397,...,15.8,47.1,28.1,27.1,14.1,37.1,24.5,"(CONTRACTS OF CAD 100,000)",F10,Combined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23666,BBG COMMODITY - CHICAGO BOARD OF TRADE,260127,2026-01-27,221602,CBT,0,221,188271,125275,172650,...,89.4,89.8,96.8,82.8,89.4,87.9,95.4,($100 X INDEX),F90,Combined
23667,BBG COMMODITY - CHICAGO BOARD OF TRADE,260120,2026-01-20,221602,CBT,0,221,189350,125416,171958,...,88.5,89.7,96.0,82.6,88.5,87.7,94.8,($100 X INDEX),F90,Combined
23668,BBG COMMODITY - CHICAGO BOARD OF TRADE,260113,2026-01-13,221602,CBT,0,221,189092,125364,171894,...,88.6,89.7,96.2,82.7,88.6,87.8,95.0,($100 X INDEX),F90,Combined
23669,BBG COMMODITY - CHICAGO BOARD OF TRADE,260106,2026-01-06,221602,CBT,0,221,204353,124995,187085,...,90.5,88.1,96.5,77.6,90.5,86.5,95.0,($100 X INDEX),F90,Combined


In [79]:
df = cot_raw[cot_raw['Report_Date_as_MM_DD_YYYY'] == '2020-03-17' ]
df = clean_financial_cot_data(df, INSTRUMENT_MAP)
df

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/96025408.py:33: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/96025408.py:63: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,...,Lev_Money_Positions_Spread_All,Other_Rept_Positions_Long_All,Other_Rept_Positions_Short_All,Other_Rept_Positions_Spread_All,NonRept_Positions_Long_All,NonRept_Positions_Short_All,Conc_Gross_LE_4_TDR_Long_All,Conc_Gross_LE_4_TDR_Short_All,Conc_Gross_LE_8_TDR_Long_All,Conc_Gross_LE_8_TDR_Short_All
0,1 Month US,2020-03-17,362970,33374,147653,22048,16171,46427,3524,234515,...,42466,10521,295,277,74,2373,64.400000,30.100000,79.600000,45.70000
1,10Y,2020-03-17,862156,73649,77277,4514,381657,262690,117532,101020,...,21324,51893,34436,0,110567,172800,29.300000,24.300000,46.700000,35.10000
2,20Y,2020-03-17,1633693,10848,198181,108222,605428,257259,279414,26948,...,329612,64738,30532,33736,174747,140181,19.100000,18.500000,28.900000,30.50000
3,2Y,2020-03-17,3308468,62201,418517,21169,1472248,725630,272455,811551,...,159939,313333,107587,4978,190594,103979,19.800000,30.500000,30.600000,44.80000
4,3 Month US,2020-03-17,245220,9575,103443,36318,5510,33794,8211,155495,...,28103,832,0,918,258,2844,73.200000,57.900000,86.500000,74.30000
5,30Y,2020-03-17,1120338,35553,231713,217,634694,77380,186099,49898,...,12196,55428,5665,0,146252,93421,25.300000,34.100000,36.200000,50.80000
6,5Y,2020-03-17,5293452,89569,622700,193571,2288778,685583,715496,713596,...,706311,239914,249666,22964,323252,398655,28.700000,20.700000,40.800000,29.80000
7,7Y,2020-03-17,5439304,102249,315179,513530,1524793,791172,964166,468891,...,1120927,140613,253061,167671,436463,368867,15.900000,15.500000,24.400000,24.70000
8,BTC,2020-03-17,4559,336,0,0,101,465,0,1869,...,55,589,504,93,1516,540,46.300000,52.100000,54.100000,71.40000
9,CAD,2020-03-17,212290,39995,27211,27358,35442,36670,8581,33360,...,23281,12323,2235,53,31897,45869,29.200000,30.600000,41.500000,40.50000


In [80]:
dv01_map = {
    "1 Month US": 41.67,   # 1M SOFR — exact
    "3 Month US": 25.00,   # 3M SOFR — exact
    "2Y":          40.0,   # ZT
    "5Y":          45.0,   # ZF
    "7Y":          65.0,   # ZN ("10Y Note", 7Y duration)
    "10Y":         95.0,   # TN (Ultra 10Y, true 10Y)
    "20Y":        150.0,   # ZB (Classic Bond)
    "30Y":        210.0,   # UB (Ultra Bond)
}
df["dv01_per_contract"] = df["Instrument"].map(dv01_map)
df = df.dropna()
df

,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,...,Other_Rept_Positions_Long_All,Other_Rept_Positions_Short_All,Other_Rept_Positions_Spread_All,NonRept_Positions_Long_All,NonRept_Positions_Short_All,Conc_Gross_LE_4_TDR_Long_All,Conc_Gross_LE_4_TDR_Short_All,Conc_Gross_LE_8_TDR_Long_All,Conc_Gross_LE_8_TDR_Short_All,dv01_per_contract
0,1 Month US,2020-03-17,362970,33374,147653,22048,16171,46427,3524,234515,...,10521,295,277,74,2373,64.4,30.1,79.6,45.7,41.67
1,10Y,2020-03-17,862156,73649,77277,4514,381657,262690,117532,101020,...,51893,34436,0,110567,172800,29.3,24.3,46.7,35.1,95.00
2,20Y,2020-03-17,1633693,10848,198181,108222,605428,257259,279414,26948,...,64738,30532,33736,174747,140181,19.1,18.5,28.9,30.5,150.00
3,2Y,2020-03-17,3308468,62201,418517,21169,1472248,725630,272455,811551,...,313333,107587,4978,190594,103979,19.8,30.5,30.6,44.8,40.00
4,3 Month US,2020-03-17,245220,9575,103443,36318,5510,33794,8211,155495,...,832,0,918,258,2844,73.2,57.9,86.5,74.3,25.00
5,30Y,2020-03-17,1120338,35553,231713,217,634694,77380,186099,49898,...,55428,5665,0,146252,93421,25.3,34.1,36.2,50.8,210.00
6,5Y,2020-03-17,5293452,89569,622700,193571,2288778,685583,715496,713596,...,239914,249666,22964,323252,398655,28.7,20.7,40.8,29.8,45.00
7,7Y,2020-03-17,5439304,102249,315179,513530,1524793,791172,964166,468891,...,140613,253061,167671,436463,368867,15.9,15.5,24.4,24.7,65.00


In [81]:
# Compute net positions per category (long - short)
df["Dealer_Net"]   = df["Dealer_Positions_Long_All"]   - df["Dealer_Positions_Short_All"]
df["AssetMgr_Net"] = df["Asset_Mgr_Positions_Long_All"] - df["Asset_Mgr_Positions_Short_All"]
df["LevFund_Net"]  = df["Lev_Money_Positions_Long_All"] - df["Lev_Money_Positions_Short_All"]
# Adjust column names above to match your actual df columns — check with df.columns

# Convert to DV01-weighted positioning ($ per bp)
df["Dealer_DV01"]   = df["Dealer_Net"]   * df["dv01_per_contract"]
df["AssetMgr_DV01"] = df["AssetMgr_Net"] * df["dv01_per_contract"]
df["LevFund_DV01"]  = df["LevFund_Net"]  * df["dv01_per_contract"]

#Opne inrterst for all
df["Dealer_All"]   = df["Dealer_Positions_Long_All"]   + df["Dealer_Positions_Short_All"]
df["AssetMgr_All"] = df["Asset_Mgr_Positions_Long_All"] + df["Asset_Mgr_Positions_Short_All"]
df["LevFund_All"]  = df["Lev_Money_Positions_Long_All"] + df["Lev_Money_Positions_Short_All"]


#Opne inrterst for all
df["Dealer_Porportion_Of_OI"]   = df["Dealer_All"]/(2*df['Open_Interest_All'])
df["AssetMgr_Porportion_Of_OI"] = df["AssetMgr_All"]/(2*df['Open_Interest_All'])
df["LevFund_Porportion_Of_OI"]  = df["LevFund_All"] /(2*df['Open_Interest_All'])

df["Dealer_Net_pct_oi"]   = (df["Dealer_Positions_Long_All"]   - df["Dealer_Positions_Short_All"])/df['Open_Interest_All']
df["AssetMgr_Net_pct_oi"] = (df["Asset_Mgr_Positions_Long_All"] - df["Asset_Mgr_Positions_Short_All"])/df['Open_Interest_All']
df["LevFund_Net_pct_oi"]  = (df["Lev_Money_Positions_Long_All"] - df["Lev_Money_Positions_Short_All"])/df['Open_Interest_All']

# Rates-only subset for the term-structure chart
rates_order = ["1 Month US", "3 Month US", "2Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
rates_df = df[df["Instrument"].isin(rates_order)].copy()
rates_df["Instrument"] = pd.Categorical(rates_df["Instrument"],
                                         categories=rates_order, ordered=True)
rates_df = rates_df.sort_values("Instrument")
rates_df

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/2277328273.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/2277328273.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_18510/2277328273.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,...,LevFund_DV01,Dealer_All,AssetMgr_All,LevFund_All,Dealer_Porportion_Of_OI,AssetMgr_Porportion_Of_OI,LevFund_Porportion_Of_OI,Dealer_Net_pct_oi,AssetMgr_Net_pct_oi,LevFund_Net_pct_oi
0,1 Month US,2020-03-17,362970,33374,147653,22048,16171,46427,3524,234515,...,5692455.36,181027,62598,332422,0.249369,0.086230,0.457919,-0.314844,-0.083357,0.376362
4,3 Month US,2020-03-17,245220,9575,103443,36318,5510,33794,8211,155495,...,3097675.00,113018,39304,187083,0.230442,0.080140,0.381460,-0.382791,-0.115341,0.505289
3,2Y,2020-03-17,3308468,62201,418517,21169,1472248,725630,272455,811551,...,-27306520.00,480718,2197878,2305765,0.072650,0.332159,0.348464,-0.107698,0.225669,-0.206338
6,5Y,2020-03-17,5293452,89569,622700,193571,2288778,685583,715496,713596,...,-44320950.00,712269,2974361,2412102,0.067278,0.280947,0.227838,-0.100715,0.302864,-0.186062
7,7Y,2020-03-17,5439304,102249,315179,513530,1524793,791172,964166,468891,...,-30929665.00,417428,2315965,1413623,0.038371,0.212892,0.129945,-0.039147,0.134874,-0.087482
1,10Y,2020-03-17,862156,73649,77277,4514,381657,262690,117532,101020,...,-6703485.00,150926,644347,272603,0.087528,0.373684,0.158094,-0.004208,0.137988,-0.081845
2,20Y,2020-03-17,1633693,10848,198181,108222,605428,257259,279414,26948,...,-34441200.00,209029,862687,283504,0.063974,0.264030,0.086768,-0.114668,0.213118,-0.140545
5,30Y,2020-03-17,1120338,35553,231713,217,634694,77380,186099,49898,...,-97387290.00,267266,712074,563545,0.119279,0.317794,0.251507,-0.175090,0.497452,-0.413937


In [82]:
# Duration in years for x-axis positioning
duration_years = {
    "1 Month US": 1/12, "3 Month US": 0.25,
    "2Y": 2, "5Y": 5, "7Y": 7, "10Y": 10, "20Y": 20, "30Y": 30
}
rates_df["duration_years"] = rates_df["Instrument"].map(duration_years)

In [83]:
rates_df = rates_df.iloc[2:]
rates_df

,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Dealer_Positions_Spread_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Asset_Mgr_Positions_Spread_All,Lev_Money_Positions_Long_All,...,Dealer_All,AssetMgr_All,LevFund_All,Dealer_Porportion_Of_OI,AssetMgr_Porportion_Of_OI,LevFund_Porportion_Of_OI,Dealer_Net_pct_oi,AssetMgr_Net_pct_oi,LevFund_Net_pct_oi,duration_years
3,2Y,2020-03-17,3308468,62201,418517,21169,1472248,725630,272455,811551,...,480718,2197878,2305765,0.072650,0.332159,0.348464,-0.107698,0.225669,-0.206338,2.0
6,5Y,2020-03-17,5293452,89569,622700,193571,2288778,685583,715496,713596,...,712269,2974361,2412102,0.067278,0.280947,0.227838,-0.100715,0.302864,-0.186062,5.0
7,7Y,2020-03-17,5439304,102249,315179,513530,1524793,791172,964166,468891,...,417428,2315965,1413623,0.038371,0.212892,0.129945,-0.039147,0.134874,-0.087482,7.0
1,10Y,2020-03-17,862156,73649,77277,4514,381657,262690,117532,101020,...,150926,644347,272603,0.087528,0.373684,0.158094,-0.004208,0.137988,-0.081845,10.0
2,20Y,2020-03-17,1633693,10848,198181,108222,605428,257259,279414,26948,...,209029,862687,283504,0.063974,0.264030,0.086768,-0.114668,0.213118,-0.140545,20.0
5,30Y,2020-03-17,1120338,35553,231713,217,634694,77380,186099,49898,...,267266,712074,563545,0.119279,0.317794,0.251507,-0.175090,0.497452,-0.413937,30.0


In [72]:
rates_df.columns.values.tolist()

['Instrument',
 'Report_Date_as_MM_DD_YYYY',
 'Open_Interest_All',
 'Dealer_Positions_Long_All',
 'Dealer_Positions_Short_All',
 'Dealer_Positions_Spread_All',
 'Asset_Mgr_Positions_Long_All',
 'Asset_Mgr_Positions_Short_All',
 'Asset_Mgr_Positions_Spread_All',
 'Lev_Money_Positions_Long_All',
 'Lev_Money_Positions_Short_All',
 'Lev_Money_Positions_Spread_All',
 'Other_Rept_Positions_Long_All',
 'Other_Rept_Positions_Short_All',
 'Other_Rept_Positions_Spread_All',
 'NonRept_Positions_Long_All',
 'NonRept_Positions_Short_All',
 'Conc_Gross_LE_4_TDR_Long_All',
 'Conc_Gross_LE_4_TDR_Short_All',
 'Conc_Gross_LE_8_TDR_Long_All',
 'Conc_Gross_LE_8_TDR_Short_All',
 'dv01_per_contract',
 'Dealer_Net',
 'AssetMgr_Net',
 'LevFund_Net',
 'Dealer_DV01',
 'AssetMgr_DV01',
 'LevFund_DV01',
 'Dealer_All',
 'AssetMgr_All',
 'LevFund_All',
 'Dealer_Porportion_Of_OI',
 'AssetMgr_Porportion_Of_OI',
 'LevFund_Porportion_Of_OI',
 'Dealer_Net_pct_oi',
 'AssetMgr_Net_pct_oi',
 'LevFund_Net_pct_oi',
 'duratio

In [84]:
px.line(rates_df, x = 'duration_years', y = ['Dealer_Net_pct_oi', 'AssetMgr_Net_pct_oi', 'LevFund_Net_pct_oi' ] )

In [85]:
px.line(rates_df, x = 'duration_years', y = ['Dealer_Porportion_Of_OI', 'AssetMgr_Porportion_Of_OI', 'LevFund_Porportion_Of_OI' ] )

In [86]:
px.bar(rates_df, x = "duration_years", y = "Open_Interest_All")

In [ ]:
px.bar(rates_df, x = "duration_years", y = "Open_Interest_All")